# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIRⁿ dataset using the `mlcroissant` library. The notebook demonstrates each step to inspect dataset structure and records using the Croissant schema.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs, the fields, and columns. Each entity in the Croissant dataset is referenced by its `@id`.

In [ ]:
# List all available record sets and their @id and description
print('Available record sets:')
for record_set in metadata.record_sets:  # metadata.record_sets gives list of croissant.RecordSet
    print(f"  RecordSet name: {record_set.name}, @id: {record_set.id}")
    if hasattr(record_set, 'description') and record_set.description:
        print(f"    Description: {record_set.description}")
    # List fields for this record set
    print('    Fields:')
    for field in record_set.fields:
        print(f"      {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print('    Columns:')
    for column in getattr(record_set, 'columns', []):  # Defensive in case columns attribute is absent
        print(f"      Column @id: {column.id}, field: {column.field_id}, source: {getattr(column, 'source', None)}")
    print('-'*50)

## 3. Data Extraction
Load all available record sets into DataFrames. For demonstration, we select the first record set. All variables use the record set and field `@id` to reference entities, ensuring consistent and reproducible data extraction.

In [ ]:
# Extract data from all available record sets
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id} shape: {df.shape}")

# Choose a primary record set to explore (first one for demonstration):
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]
    print(f"\nMain Record Set Columns (@id: {main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No record sets available in this dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All references use the field and record set `@id` as per Croissant specification.

In [ ]:
import numpy as np
df = dataframes[main_record_set_id]

# Inspect numeric fields by Croissant field @id
numeric_field_id = None
for record_set in metadata.record_sets:
    if record_set.id == main_record_set_id:
        for field in record_set.fields:
            if field.data_type in ['Float', 'Integer', 'Number']:
                if field.id in df.columns:
                    numeric_field_id = field.id
                    print(f"Found numeric field: {field.name} (@id: {field.id}) of type {field.data_type}")
                    break
        break

if numeric_field_id is None:
    raise ValueError('No numeric field found in the record set for demonstration.')

# Filter out records with value greater than a threshold and normalize
threshold = df[numeric_field_id].quantile(0.5) if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (median): {filtered_df.shape[0]} rows")

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

# Try grouping by the first non-numeric field
group_field_id = None
for record_set in metadata.record_sets:
    if record_set.id == main_record_set_id:
        for field in record_set.fields:
            if field.data_type not in ['Float', 'Integer', 'Number'] and field.id in df.columns:
                group_field_id = field.id
                print(f"Will group analysis by: {field.name} (@id: {field.id})")
                break
        break

if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:\n", grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot a histogram of the numeric field and a bar plot if grouping is possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped data is available, plot group mean
if 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Group-wise mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=60, ha='right')
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load metadata and records from the FAIRⁿ mass spectrometry dataset via the Croissant schema using only `@id` identifiers,
- Inspect and list the record set, field, and column structure,
- Load all records dynamically and select fields for data processing,
- Carry out basic data filtering, normalization, and group-based aggregation using only entity `@id`s,
- Visualize numeric distributions and group statistics.

To investigate the data more deeply, inspect the available field and record set `@id`s in your loaded metadata for further custom analyses.
